# 05 — Rhetorical-Strategy Scoring

Scores every comment on a set of named rhetorical strategies, using a large language model, following the general approach of Multi-Strategy Persuasion Scoring (MS-PS) — asking an LLM to rate a text's use of specific, named persuasion strategies rather than only classifying it directly (Labruna, Modzelewski, Satta & Da San Martino, *Detecting Winning Arguments with Large Language Models and Persuasion Strategies*, Findings of EACL 2026).

That paper's own six strategies (Attack on Reputation, Justification, Simplification, Distraction, Call, Manipulative Wording) were built for propaganda-technique detection on news articles and for comparing pairs of arguments head-to-head. This notebook adapts the same *method* — an LLM assigning a numeric score per named strategy — to a different list of eight strategies, chosen instead for the kind of engagement that plausibly earns a delta in this dataset: constructive argumentative moves rather than propaganda techniques.

## Strategy Rubric

Each comment is scored on all eight strategies independently, on a 1-5 scale (a comment can score high on several strategies at once, or none at all). A 1-5 scale is used rather than a wider range specifically to keep both LLM scoring and the human validation pass (further below) practical to apply consistently.

1. **Justification** — gives a reason or logical basis for the position taken, beyond simply asserting it.
   *1 = bare assertion, no reason given. 5 = the position is built substantially on stated reasoning, with a clear link between the reason and the claim.*

2. **Concession** — acknowledges a valid point, limitation, or partial truth in the opposing view before responding to it.
   *1 = no acknowledgment of the opposing view's validity. 5 = a substantial, specific concession that engages with a real strength of the opposing view.*

3. **Evidence** — cites a concrete fact, statistic, study, source, or specific example to support a claim.
   *1 = no supporting evidence beyond the writer's own assertion. 5 = multiple concrete, specific pieces of evidence.*

4. **Refutation** — directly identifies and counters a specific flaw, assumption, or weak point in the original argument.
   *1 = no direct engagement with a specific point in the original argument. 5 = precise identification and rebuttal of one or more specific flaws or assumptions.*

5. **Personal narrative** — draws on the commenter's own lived experience to make the case.
   *1 = no personal experience invoked. 5 = the argument is substantially grounded in specific, detailed personal experience.*

6. **Call-to-action** — offers a concrete alternative way to act on or think about the issue, not just criticism of the original view.
   *1 = no alternative action or framing offered. 5 = a clear, concrete alternative way to act on or think about the issue.*

7. **Simplification** — uses an analogy, thought experiment, or simplified restatement to make an abstract point concrete.
   *1 = no analogy or simplifying device used. 5 = the argument's core point rests on a well-constructed analogy or thought experiment.*

8. **Question-asking** — poses a genuine question back to the original poster that probes an assumption or invites clarification. Does not include a rhetorical question the author poses and immediately answers themselves - that's a stylistic device, not an invitation for the original poster to respond.
   *1 = no genuine questions posed. 5 = the argument is substantially built around probing questions that invite reconsidering a specific assumption.*

Across all eight, a score of 3 marks the strategy being clearly present but modest in scope or development — the midpoint is "present," not "absent" or "dominant."

## Scoring Process

This section develops and validates the scoring prompt before running it across the full dataset: piloted on a small sample of real working-set comments, revised based on what the pilot showed (including a redesign to send the original post's text, not just the comment - see below), then run in full and checked with a retest and hand-scoring pass.

The pilot is batched (multiple comments per call, matching the pattern already used throughout `03.1`-`03.3`) and checkpointed, so a re-run only fills in rows that don't have scores yet rather than re-scoring everything.

The scores are validated separately and more formally afterward, using a held-out sample of comments the 30-word floor already excludes from modeling - a retest pass (re-running the same prompt on the same comments) and a small hand-scored comparison.

### Initial Prompt Draft

Batched, one line of eight comma-separated scores per comment - matching the format `common_functions.py`'s existing Gemini prompts already use for `sentiment`/`tone`/`style`/`ethos,pathos,logos`. Reusing that pattern means reusing the existing batching, checkpointing, and retry plumbing directly, and it keeps each API call scoring ~20 comments instead of 1 - a meaningful cost difference at this dataset's size.

In [1]:
STRATEGY_LLM_PROMPT_PREFIX = """Rate each of the following numbered Reddit comments on eight rhetorical strategies, each on a scale from 1 (not present) to 5 (strongly and substantially present). Score each strategy independently - a comment can score high on several strategies at once, or low on all of them.

- justification: gives a reason or logical basis for the position taken, beyond simply asserting it
- concession: acknowledges a valid point, limitation, or partial truth in the opposing view before responding to it
- evidence: cites a concrete fact, statistic, study, source, or specific example to support a claim
- refutation: directly identifies and counters a specific flaw, assumption, or weak point in the original argument
- personal_narrative: draws on the commenter's own lived experience to make the case
- call_to_action: offers a concrete alternative way to act on or think about the issue, not just criticism of the original view
- simplification: uses an analogy, thought experiment, or simplified restatement to make an abstract point concrete
- question_asking: poses a genuine question back to the original poster that probes an assumption or invites clarification - does not include a rhetorical question the author poses and immediately answers themselves

A score of 3 means the strategy is clearly present but modest in scope or development - not "absent" and not "dominant."

Reply with exactly one line per text, in the format "N: justification,concession,evidence,refutation,personal_narrative,call_to_action,simplification,question_asking" (matching the numbers below, eight comma-separated integers in that exact order), and nothing else.

Texts:
"""

print(STRATEGY_LLM_PROMPT_PREFIX)

Rate each of the following numbered Reddit comments on eight rhetorical strategies, each on a scale from 1 (not present) to 5 (strongly and substantially present). Score each strategy independently - a comment can score high on several strategies at once, or low on all of them.

- justification: gives a reason or logical basis for the position taken, beyond simply asserting it
- concession: acknowledges a valid point, limitation, or partial truth in the opposing view before responding to it
- evidence: cites a concrete fact, statistic, study, source, or specific example to support a claim
- refutation: directly identifies and counters a specific flaw, assumption, or weak point in the original argument
- personal_narrative: draws on the commenter's own lived experience to make the case
- call_to_action: offers a concrete alternative way to act on or think about the issue, not just criticism of the original view
- simplification: uses an analogy, thought experiment, or simplified restate

### Selecting Pilot Comments

The pilot's purpose is to check whether the prompt actually discriminates between strategies, not to be a statistically representative sample - a handful of random comments risks landing in the same "medium, unremarkable" bucket for every strategy, which wouldn't reveal much. Instead, comments are deliberately pulled from several buckets, using existing engineered features as rough proxies where one exists (high `evidence_count` for `evidence`, high `logos_llm` for `justification`/`refutation`, high `pathos_llm` for `personal_narrative`), a simple text search for `question_asking` on `final_comment` rather than `cleaned_final_comment` (the latter strips all punctuation, including question marks, for its TF-IDF/word-count role - the LLM scoring itself should also read `final_comment`, not the stripped field, since punctuation and sentence structure carry real signal for a strategy like this one), length extremes as a stress test, and a random baseline for comments with no particular expected signal. `concession`, `call_to_action`, and `simplification` have no proxy available and rely on the random draws and reading the pilot output directly. All candidates come from the same population the full run will eventually score (the `word_count >= 30` floor, plus all convincing comments regardless of length), so pilot behavior should generalize to the real run.

In [2]:
import pandas as pd

comments_df = pd.read_csv('../results/04_cmv_comments_df.csv.zip')
eligible = comments_df[(comments_df['word_count'] >= 30) | (comments_df['is_convincing'] == 1)].copy()

# A plain substring search for '?' can be fooled by a question mark inside a quoted
# excerpt (lines starting with '>') - that's OP's question, not the commenter's own. Only
# counting '?' in non-quoted lines finds comments where the commenter is actually asking.
def unquoted_question_count(text):
    lines = [l for l in str(text).split(chr(10)) if not l.strip().startswith('>')]
    return sum(l.count('?') for l in lines)

eligible['unquoted_q_count'] = eligible['final_comment'].apply(unquoted_question_count)
many_questions = eligible[eligible['unquoted_q_count'] >= 3].sort_values('unquoted_q_count', ascending=False)

selections = {
    'high evidence_count':          eligible.nlargest(2, 'evidence_count'),
    'high logos_llm':               eligible.nlargest(2, 'logos_llm'),
    'high pathos_llm':              eligible.nlargest(2, 'pathos_llm'),
    '3+ unquoted question marks':   many_questions.head(2),
    'shortest eligible':            eligible.nsmallest(2, 'word_count'),
    'longest':                      eligible.nlargest(2, 'word_count'),
    'random baseline':              eligible.sample(3, random_state=42),
}

seen_idx, rows = set(), []
for reason, subset in selections.items():
    for idx in subset.index:
        if idx in seen_idx:
            continue
        seen_idx.add(idx)
        rows.append((idx, reason))

pilot_df = eligible.loc[[idx for idx, _ in rows]].copy()
pilot_df['selection_reason'] = [reason for _, reason in rows]

print(f'Pilot sample: {len(pilot_df)} comments (from {sum(len(s) for s in selections.values())} selected before de-duplication)')
pilot_df[['selection_reason', 'word_count', 'is_convincing', 'evidence_count', 'logos_llm', 'pathos_llm']]

Pilot sample: 14 comments (from 15 selected before de-duplication)


,selection_reason,word_count,is_convincing,evidence_count,logos_llm,pathos_llm
3592,high evidence_count,1563,0,31,0.9,0.7
3745,high evidence_count,537,0,31,0.8,0.4
146,high logos_llm,536,0,25,1.0,0.2
539,high logos_llm,160,0,0,1.0,0.0
546,high pathos_llm,131,0,0,1.0,1.0
547,high pathos_llm,219,1,0,1.0,1.0
3413,3+ unquoted question marks,860,1,3,0.4,1.0
3191,3+ unquoted question marks,686,0,16,0.9,0.4
3054,shortest eligible,2,1,0,0.1,0.0
1093,shortest eligible,5,1,0,0.4,0.3


### Running the Pilot

Scores the pilot comments with the draft prompt above, using `final_comment` (not `cleaned_final_comment`) as the text passed to the model. Batched and checkpointed the same way `03.1`-`03.3` score other Gemini features - progress is saved after every batch, and re-running the cell only scores rows that don't have scores yet, so a transient API failure partway through doesn't lose completed work. Each row's checkpointed scores are tagged with a hash of the prompt that produced them, so editing the prompt automatically invalidates old scores instead of silently mixing them with new ones. Read the output below for: strategies that seem confused with each other, scores that cluster on the same value regardless of what the comment actually contains, or lines that don't parse - not for any particular numeric target.

In [3]:
import re
import common_functions

genai_client = common_functions.make_gemini_client()
GEMINI_MODEL = common_functions.DEFAULT_GEMINI_MODEL

print(f"Using Gemini model: {GEMINI_MODEL}")

Using Gemini model: gemini-flash-latest


In [4]:
STRATEGY_NAMES = [
    'justification', 'concession', 'evidence', 'refutation',
    'personal_narrative', 'call_to_action', 'simplification', 'question_asking',
]

def get_strategy_scores_batch(rows):
    texts = [row['final_comment'] for row in rows]
    numbered_texts = "\n\n".join(f"{i}:\n{text}" for i, text in enumerate(texts))
    user_prompt = STRATEGY_LLM_PROMPT_PREFIX + numbered_texts
    response = common_functions.generate_content_with_retry(genai_client, GEMINI_MODEL, contents=user_prompt)

    pattern = r'\s*(\d+)\s*[:\-]\s*' + r',\s*'.join([r'(\d+)'] * len(STRATEGY_NAMES))
    scores = [None] * len(texts)
    for line in response.text.strip().splitlines():
        match = re.match(pattern, line)
        if match:
            i = int(match.group(1))
            if 0 <= i < len(scores):
                scores[i] = tuple(int(match.group(k)) for k in range(2, 2 + len(STRATEGY_NAMES)))
    return scores

In [5]:
import hashlib
import os

PILOT_CHECKPOINT_PATH = '../results/pilot_strategy_scores_checkpoint.csv'

### Pilot Results With Text

The table below shows every scored pilot row with a short preview of its text, so no row's score is disconnected from what it was scored on. To read any one row's full text alongside its full scores, use its index in the self-contained cell further below - it reloads the checkpoint itself, so it can be run on its own without running anything above it first.

In [6]:
pilot_results = pd.read_csv(PILOT_CHECKPOINT_PATH, index_col=0)

def preview(text, n=150):
    text = str(text).replace('\n', ' ')
    return text[:n] + ('...' if len(text) > n else '')

pilot_summary = pilot_results[['selection_reason'] + STRATEGY_NAMES].copy()
pilot_summary.insert(1, 'text_preview', pilot_results['final_comment'].apply(preview))
pilot_summary

,selection_reason,text_preview,justification,concession,evidence,refutation,personal_narrative,call_to_action,simplification,question_asking
3592,high evidence_count,I know this is a long post but I implore you t...,5,3,5,5,1,3,3,2
3745,high evidence_count,"> AI and Robots are only cheaper ""in relation ...",5,2,2,4,1,1,4,3
146,high logos_llm,>I guess you could say black cops are racist a...,4,2,5,5,1,1,3,4
539,high logos_llm,The low hanging issue is that these news netwo...,3,2,4,3,1,3,1,1
546,high pathos_llm,Well there's that whole first amendment thing ...,3,2,2,3,1,3,2,1
547,high pathos_llm,"All of us have an innate gender identity, you ...",4,2,2,3,1,1,3,1
3413,3+ unquoted question marks,"Los Angeles is the greatest city in the world,...",3,4,4,3,4,3,2,5
3191,3+ unquoted question marks,"I appreciate your feedback, but the ""convincin...",5,4,5,4,1,3,3,1
3054,shortest eligible,Reenactment organizers.,1,1,1,1,1,1,1,1
1093,shortest eligible,Counter argument; Chris Brown &#x200B;,1,1,2,2,1,1,1,1


In [7]:
import pandas as pd

STRATEGY_NAMES = ['justification', 'concession', 'evidence', 'refutation',
                   'personal_narrative', 'call_to_action', 'simplification', 'question_asking']
pilot_results = pd.read_csv('../results/pilot_strategy_scores_checkpoint.csv', index_col=0)

def show_pilot_comment(idx):
    row = pilot_results.loc[idx]
    print(f"Row {idx} - {row['selection_reason']} ({int(row['word_count'])} words)")
    print('-' * 70)
    print(row['final_comment'])
    print('-' * 70)
    print({name: int(row[name]) for name in STRATEGY_NAMES})

show_pilot_comment(1184)  # replace with any row index from the table above

Row 1184 - longest (1573 words)
----------------------------------------------------------------------
Hi!

Thanks for posting. This is an interesting question that comes up in many civil rights movements, I think. And that’s what I’ll try to draw a parallel to. 

For context I’m a white CIS male from Scandinavia. I’ve been an ally all my life and anti racism, feminism and HBTQ rights have been the center of my political identity since I was a teenager. 

I lesson I learned from the women’s right movement might give you some food for thought in your context. 

You probably heard the term “yes ALL men” when it comes to questions of patriarchy in general and sexual abuse in particular. 

This was a statement I self struggled with a lot in my twenties. Maybe in a way similar to how you struggle with “All white people” now. 

I didn’t like getting associated with what I was actively fighting against and trying to change. I didn’t like the prejudice I felt I could get met with in feminist c

### Checking the call_to_action Ceiling

No comment in the 14-row pilot scored above 3 on `call_to_action`, even the longest and most substantive ones. A keyword search of the dataset for explicit suggestion language ("you should", "I suggest", "instead you", etc.) mostly turned up false positives - quoted excerpts from OP, or rhetorical framing rather than a genuine proposed action - suggesting CMV comments may just rarely contain a concrete, prescriptive "do X instead" (the sub is about debating a view, not giving advice). But that's hard to tell apart from the model/prompt simply not using the top of the scale for this strategy.

To separate the two explanations, two synthetic texts (not from the dataset, not saved anywhere) are scored with the same prompt: one written to unambiguously warrant a high `call_to_action` score, and a minimal baseline with no content at all.

In [8]:
synthetic_rows = [
    {'final_comment': (
        "Instead of banning cars downtown outright, you should try converting one lane to a "
        "protected bike lane for a six-month trial and measure the traffic impact before making "
        "it permanent. Start with the two blocks near the transit station, publish weekly "
        "ridership counts, and expand only if ridership rises by 20 percent."
    )},
    {'final_comment': "I disagree."},
]

try:
    synthetic_scores = get_strategy_scores_batch(synthetic_rows)
    for row, score_tuple in zip(synthetic_rows, synthetic_scores):
        print(row['final_comment'][:80] + ('...' if len(row['final_comment']) > 80 else ''))
        print(dict(zip(STRATEGY_NAMES, score_tuple)))
        print()
except Exception as e:
    print(f'synthetic scoring failed, re-run this cell later: {e}')

Instead of banning cars downtown outright, you should try converting one lane to...
{'justification': 3, 'concession': 1, 'evidence': 1, 'refutation': 2, 'personal_narrative': 1, 'call_to_action': 5, 'simplification': 1, 'question_asking': 1}

I disagree.
{'justification': 1, 'concession': 1, 'evidence': 1, 'refutation': 1, 'personal_narrative': 1, 'call_to_action': 1, 'simplification': 1, 'question_asking': 1}



### v2: Grouping Batches by Original Post

The prompt above never sent the original post's text at all - only whatever the commenter happened to quote inline. But `concession` and `refutation` are both defined relative to the original post's argument, so the model needs to actually see it, not infer it from partial quotes.

Sending the full original post with every single comment would roughly double token usage per item. Instead, batches are grouped by original post: comments replying to the same post go into the same batch, with the post included once and each comment numbered underneath it - the cost scales with the number of distinct threads touched, not the number of comments. Where a comment has a `thread_text` (the same commenter's own earlier turns in this reply chain, present for about a third of rows), that's included immediately before the final comment it belongs to, so the model can see the commenter's own build-up, not just their last line.

This pilot's 14 rows span 13 distinct original posts (deliberately picked for diversity, not shared threads), so the token savings barely show up here - they matter at the scale of the real run, where the working dataset's ~2,753 comments come from only ~401 threads. Scored to a separate checkpoint (`pilot_strategy_scores_op_checkpoint.csv`), so the original per-comment pilot results above stay intact as a fallback if this version doesn't look better.

In [9]:
STRATEGY_LLM_PROMPT_PREFIX_V2 = """Rate each of the following numbered replies to the original post below, on eight rhetorical strategies, each on a scale from 1 (not present) to 5 (strongly and substantially present). Score each strategy independently - a reply can score high on several strategies at once, or low on all of them. Some replies include the same author's own earlier turns in this conversation before their final reply - score only their final reply (the last part of the text), using the earlier turns only as context for what they're responding to.

- justification: gives a reason or logical basis for the position taken, beyond simply asserting it
- concession: acknowledges a valid point, limitation, or partial truth in the original post's view before responding to it
- evidence: cites a concrete fact, statistic, study, source, or specific example to support a claim
- refutation: directly identifies and counters a specific flaw, assumption, or weak point in the original post's argument
- personal_narrative: draws on the commenter's own lived experience to make the case
- call_to_action: offers a concrete alternative way to act on or think about the issue, not just criticism of the original post
- simplification: uses an analogy, thought experiment, or simplified restatement to make an abstract point concrete
- question_asking: poses a genuine question back to the original poster that probes an assumption or invites clarification - does not include a rhetorical question the author poses and immediately answers themselves

A score of 3 means the strategy is clearly present but modest in scope or development - not "absent" and not "dominant."

Reply with exactly one line per numbered reply, in the format "N: justification,concession,evidence,refutation,personal_narrative,call_to_action,simplification,question_asking" (matching the numbers below, eight comma-separated integers in that exact order), and nothing else.

Original post:
"""

print(STRATEGY_LLM_PROMPT_PREFIX_V2)

Rate each of the following numbered replies to the original post below, on eight rhetorical strategies, each on a scale from 1 (not present) to 5 (strongly and substantially present). Score each strategy independently - a reply can score high on several strategies at once, or low on all of them. Some replies include the same author's own earlier turns in this conversation before their final reply - score only their final reply (the last part of the text), using the earlier turns only as context for what they're responding to.

- justification: gives a reason or logical basis for the position taken, beyond simply asserting it
- concession: acknowledges a valid point, limitation, or partial truth in the original post's view before responding to it
- evidence: cites a concrete fact, statistic, study, source, or specific example to support a claim
- refutation: directly identifies and counters a specific flaw, assumption, or weak point in the original post's argument
- personal_narrative: 

In [10]:
def get_strategy_scores_batch_op(original_post, rows):
    texts = []
    for row in rows:
        parts = []
        if isinstance(row.get('thread_text'), str) and row['thread_text'].strip():
            parts.append(row['thread_text'])
        parts.append(row['final_comment'])
        texts.append('\n\n'.join(parts))

    numbered_texts = '\n\n'.join(f'{i}:\n{text}' for i, text in enumerate(texts))
    user_prompt = STRATEGY_LLM_PROMPT_PREFIX_V2 + str(original_post) + '\n\nReplies:\n\n' + numbered_texts
    response = common_functions.generate_content_with_retry(genai_client, GEMINI_MODEL, contents=user_prompt)

    pattern = r'\s*(\d+)\s*[:\-]\s*' + r',\s*'.join([r'(\d+)'] * len(STRATEGY_NAMES))
    scores = [None] * len(texts)
    # response.text is None when Gemini blocks the whole reply (safety filtering on the
    # original post's content, not the specific comment being scored) rather than returning
    # malformed text - treat that the same as an unparseable response, not a crash.
    for line in (response.text or '').strip().splitlines():
        match = re.match(pattern, line)
        if match:
            i = int(match.group(1))
            if 0 <= i < len(scores):
                scores[i] = tuple(int(match.group(k)) for k in range(2, 2 + len(STRATEGY_NAMES)))
    return scores

In [11]:
PROMPT_HASH_V2 = hashlib.md5(STRATEGY_LLM_PROMPT_PREFIX_V2.encode()).hexdigest()[:8]
PILOT_CHECKPOINT_PATH_V2 = '../results/pilot_strategy_scores_op_checkpoint.csv'

pilot_df = common_functions.score_comments_by_thread(
    pilot_df, STRATEGY_NAMES, get_strategy_scores_batch_op, PILOT_CHECKPOINT_PATH_V2, PROMPT_HASH_V2)
pilot_df[['selection_reason'] + STRATEGY_NAMES]

Loaded checkpoint: 14 rows match the current prompt, 0 row(s) were scored under a different prompt version and will be re-scored, 0 row(s) were never scored yet
Done: 14 / 14 scored


,selection_reason,justification,concession,evidence,refutation,personal_narrative,call_to_action,simplification,question_asking
3592,high evidence_count,5.0,3.0,5.0,5.0,1.0,2.0,2.0,1.0
3745,high evidence_count,4.0,1.0,2.0,5.0,1.0,1.0,4.0,4.0
146,high logos_llm,5.0,2.0,5.0,5.0,1.0,1.0,3.0,2.0
539,high logos_llm,4.0,2.0,4.0,2.0,1.0,3.0,2.0,1.0
546,high pathos_llm,4.0,1.0,2.0,4.0,1.0,3.0,3.0,1.0
547,high pathos_llm,4.0,2.0,2.0,4.0,1.0,1.0,4.0,1.0
3413,3+ unquoted question marks,4.0,4.0,5.0,4.0,3.0,3.0,2.0,2.0
3191,3+ unquoted question marks,5.0,4.0,5.0,5.0,1.0,3.0,3.0,1.0
3054,shortest eligible,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0
1093,shortest eligible,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0


### Comparing the Two Pilot Versions

Both pilot versions are now fully scored on the same 14 rows. Comparing them directly shows whether adding real original-post context actually changed anything, rather than just assuming it would.

In [12]:
v1_results = pd.read_csv(PILOT_CHECKPOINT_PATH, index_col=0)
v2_results = pd.read_csv(PILOT_CHECKPOINT_PATH_V2, index_col=0)

diff = v2_results[STRATEGY_NAMES] - v1_results[STRATEGY_NAMES]
print('Mean absolute change per strategy (v2 - v1):')
print(diff.abs().mean().sort_values(ascending=False))
print()

context_cmp = pd.DataFrame({
    'reason': v1_results['selection_reason'],
    'concession_v1': v1_results['concession'], 'concession_v2': v2_results['concession'],
    'refutation_v1': v1_results['refutation'], 'refutation_v2': v2_results['refutation'],
    'question_asking_v1': v1_results['question_asking'], 'question_asking_v2': v2_results['question_asking'],
})
context_cmp

Mean absolute change per strategy (v2 - v1):
refutation            0.642857
question_asking       0.500000
justification         0.428571
evidence              0.285714
simplification        0.285714
concession            0.214286
call_to_action        0.142857
personal_narrative    0.071429
dtype: float64



,reason,concession_v1,concession_v2,refutation_v1,refutation_v2,question_asking_v1,question_asking_v2
3592,high evidence_count,3,3.0,5,5.0,2,1.0
3745,high evidence_count,2,1.0,4,5.0,3,4.0
146,high logos_llm,2,2.0,5,5.0,4,2.0
539,high logos_llm,2,2.0,3,2.0,1,1.0
546,high pathos_llm,2,1.0,3,4.0,1,1.0
547,high pathos_llm,2,2.0,3,4.0,1,1.0
3413,3+ unquoted question marks,4,4.0,3,4.0,5,2.0
3191,3+ unquoted question marks,4,4.0,4,5.0,1,1.0
3054,shortest eligible,1,1.0,1,2.0,1,1.0
1093,shortest eligible,1,1.0,2,2.0,1,1.0


**`refutation` moved up consistently once the model could see what was actually being refuted** - 6 of 14 rows increased (e.g. row 546: 3->4, row 3191: 4->5, row 3054: 1->2), only 1 decreased. That's the expected direction: without the original post, the model was guessing at what argument was being countered from whatever the commenter happened to quote inline; with it, `refutation` (defined relative to "the original post's argument") has something concrete to check against.

**`question_asking` moved in a more interesting way on row 3413** - a comment made up largely of questions addressed directly to the original poster ("Have you seen a show at the Hollywood Bowl? Have you had the ham and eggs at Nick's Cafe?"), which v1 scored 5 (the strongest possible signal) on the strength of that direct address alone. v2 drops it to 2. Re-reading the comment's actual original post (a general complaint that Los Angeles is a bad city) makes the v2 score look like the more defensible one, not a regression: this is a rhetorical enumeration device (listing the city's attractions persuasively), not a question that probes a specific assumption the original poster stated or invites them to clarify something. v1's score conflated "addressed to the original poster" with "genuinely probing" - without seeing what the original poster actually argued, there was no way to catch that distinction. `concession` barely moved (small ±1 shifts on 3 of 14 rows) - the original per-comment version already had partial access to the original post's argument via the commenter's own inline quoting convention, so there was less left for full context to correct.

**Decision: the OP-grouped version (v2) is what the full run uses.** It fixes a real gap (`concession`/`refutation` are explicitly defined relative to the original post's argument, which v1 never saw), the one clear behavior change it produced (`question_asking` on row 3413) looks like a correction rather than noise, and grouping batches by thread keeps the added cost proportional to the number of threads touched rather than the number of comments.

## Scoring the Full Dataset

Applies the validated OP-grouped approach above to the real working dataset: `results/04_cmv_comments_df.csv`, filtered to `word_count >= 30` (plus all convincing comments regardless of length) - the same population the pilot sample was drawn from, now scored in full rather than as a 14-row sample.

That's 2,753 comments across 400 distinct original posts (6.9 comments/thread on average, up to 37 in the largest) - 415 batches once threads larger than 20 are split into multiple chunks. Reuses the same prompt (`STRATEGY_LLM_PROMPT_PREFIX_V2`), scoring function (`get_strategy_scores_batch_op`), and resumable, checkpointed scoring loop already built and exercised above, pointed at a new checkpoint file rather than the pilot's.

Only the columns actually needed for scoring (and for identifying which comment is which) are kept in memory and checkpointed - not the full ~240-column feature matrix `04_cmv_comments_df.csv` carries, which would make every checkpoint write far larger than it needs to be for no benefit.

In [13]:
scoring_df = pd.read_csv('../results/04_cmv_comments_df.csv.zip')
scoring_df = scoring_df[(scoring_df['word_count'] >= 30) | (scoring_df['is_convincing'] == 1)].copy()
scoring_df = scoring_df[['original_post', 'thread_text', 'final_comment', 'word_count', 'is_convincing']]

print(f'{len(scoring_df)} comments across {scoring_df["original_post"].nunique()} original posts')

2753 comments across 400 original posts


In [14]:
FULL_CHECKPOINT_PATH = '../results/strategy_scores_checkpoint.csv'

scoring_df = common_functions.score_comments_by_thread(
    scoring_df, STRATEGY_NAMES, get_strategy_scores_batch_op, FULL_CHECKPOINT_PATH, PROMPT_HASH_V2)

Loaded checkpoint: 2753 rows match the current prompt, 0 row(s) were scored under a different prompt version and will be re-scored, 0 row(s) were never scored yet
Done: 2753 / 2753 scored


## Validating the Scores

The pilot above validated the *prompt design* - does the model discriminate between strategies, does it use the scale sensibly - on a deliberately hand-picked, non-representative sample, checked by reading transcripts. This section validates the *scores themselves*: are they internally consistent (test-retest), and do they agree with an independent human judgment (hand-scoring)? Both checks are scaled down from a fully rigorous study given the project's time/budget constraints - a lightweight check, not a formal reliability study.

### Test-Retest

Re-scores a random sample of already-scored comments completely independently, and compares the two passes. At `temperature=0` the model's output should be close to deterministic, but LLM APIs aren't always perfectly reproducible in practice (known nondeterminism from floating-point/batching effects on the serving side) - this checks whether that holds here, rather than assuming it. Drawn from the real working set, not a pool of shorter, excluded comments — testing reliability on a different, mostly-trivial population (near-empty comments that floor every strategy at 1) would be a weaker proxy for whether the scores actually being used are reliable.

This only checks *consistency*, not *correctness* - a model could give the same wrong answer twice and pass this check. Agreement with an independent human judgment (below) is what checks correctness.

In [15]:
RETEST_SAMPLE_SIZE = 200
retest_sample = scoring_df.sample(RETEST_SAMPLE_SIZE, random_state=7).copy()
print(f'{len(retest_sample)} comments selected for retest, from {retest_sample["original_post"].nunique()} original posts')

200 comments selected for retest, from 142 original posts


In [16]:
RETEST_CHECKPOINT_PATH = '../results/strategy_scores_retest_checkpoint.csv'

retest_scores_df = retest_sample[['original_post', 'thread_text', 'final_comment']].copy()
retest_scores_df = common_functions.score_comments_by_thread(
    retest_scores_df, STRATEGY_NAMES, get_strategy_scores_batch_op, RETEST_CHECKPOINT_PATH, PROMPT_HASH_V2)

Loaded checkpoint: 200 rows match the current prompt, 0 row(s) were scored under a different prompt version and will be re-scored, 0 row(s) were never scored yet
Done: 200 / 200 scored


In [17]:
original_scores = scoring_df.loc[retest_sample.index, STRATEGY_NAMES]
retest_scores = retest_scores_df.loc[retest_sample.index, STRATEGY_NAMES].apply(pd.to_numeric, errors='coerce')

unscored = retest_scores.isna().any(axis=1)
if unscored.any():
    print(f'{unscored.sum()} row(s) have no parseable retest response - excluded from the comparison below')
    original_scores = original_scores.loc[~unscored]
    retest_scores = retest_scores.loc[~unscored]

retest_diff = (retest_scores.astype(float) - original_scores.astype(float))

agreement = pd.DataFrame({
    'exact_match_pct': (retest_diff == 0).mean() * 100,
    'within_1_pct': (retest_diff.abs() <= 1).mean() * 100,
    'mean_abs_diff': retest_diff.abs().mean(),
})
agreement.sort_values('exact_match_pct')

1 row(s) have no parseable retest response - excluded from the comparison below


,exact_match_pct,within_1_pct,mean_abs_diff
refutation,63.819095,95.477387,0.412060
concession,72.361809,98.994975,0.286432
justification,73.366834,98.492462,0.286432
call_to_action,74.874372,95.477387,0.296482
simplification,74.874372,97.989950,0.271357
evidence,77.386935,99.497487,0.236181
personal_narrative,86.432161,99.497487,0.140704
question_asking,86.934673,97.989950,0.155779


**Strong agreement across every strategy.** All eight land at 83-99% of scores within ±1 point across the two independent passes, with mean absolute differences between 0.14 and 0.41. `refutation` is the least perfectly reproducible (64% exact match) but still 95% within ±1 - notable given it also turns out to be the hardest strategy to calibrate against human judgment (see the hand-scoring review below), suggesting it's a genuinely harder judgment call for the model, not just noisy. `personal_narrative` and `question_asking` are the most stable (86-87% exact match).

This only checks whether the model is *consistent with itself* - it says nothing about whether the scores are *correct*. That's what the hand-scoring comparison below is for.

### Hand-Scoring

Compares the LLM's scores against an independent human judgment on the same rubric - this is the half of validation that checks *correctness*, not just consistency. 15-20 comments (small on purpose, given the project's time constraints), randomly drawn from the real working set with light stratification (not the excluded short-comment pool, for the same reason as the retest sample above), excluding the 14 rows already used in the pilot - those were read and discussed at length already, so scoring them again wouldn't be a blind judgment.

**The cell below must be run interactively, not through headless execution** - it uses `input()` to collect scores one comment at a time, which has no meaning outside a live Jupyter session. It shows the original post and the comment (the same context the model had) but deliberately never shows the model's own score for that row while you're scoring, so the comparison isn't anchored to what the LLM already said. Saves to a checkpoint after every row, so it can be stopped and resumed across multiple sittings.

In [18]:
HAND_SCORE_SAMPLE_SIZE = 18
excluded_from_hand_scoring = set(pilot_results.index)  # already read closely during piloting - not a blind judgment anymore
hand_score_candidates = scoring_df[~scoring_df.index.isin(excluded_from_hand_scoring)]

# Light stratification so the sample isn't accidentally skewed toward one kind of comment -
# split into convincing/non-convincing and short/long before sampling from each group.
convincing = hand_score_candidates[hand_score_candidates['is_convincing'] == 1]
non_convincing = hand_score_candidates[hand_score_candidates['is_convincing'] == 0]
n_convincing = HAND_SCORE_SAMPLE_SIZE // 2
hand_sample = pd.concat([
    convincing.sample(n_convincing, random_state=11),
    non_convincing.sample(HAND_SCORE_SAMPLE_SIZE - n_convincing, random_state=11),
])
print(f'{len(hand_sample)} comments selected for hand-scoring '
      f'({hand_sample["is_convincing"].sum()} convincing, {(~hand_sample["is_convincing"].astype(bool)).sum()} not)')

18 comments selected for hand-scoring (9 convincing, 9 not)


In [19]:
HAND_SCORE_CHECKPOINT_PATH = '../results/hand_scores_checkpoint.csv'

if os.path.exists(HAND_SCORE_CHECKPOINT_PATH):
    hand_scores = pd.read_csv(HAND_SCORE_CHECKPOINT_PATH, index_col=0)
else:
    hand_scores = pd.DataFrame(index=hand_sample.index, columns=STRATEGY_NAMES)

for idx in hand_sample.index:
    if hand_scores.loc[idx].notna().all():
        continue  # already scored in an earlier sitting

    print('=' * 70)
    print(f'Row {idx}')
    print('=' * 70)
    print('ORIGINAL POST:')
    print(hand_sample.loc[idx, 'original_post'])
    print()
    if isinstance(hand_sample.loc[idx, 'thread_text'], str) and hand_sample.loc[idx, 'thread_text'].strip():
        print('EARLIER TURNS BY THIS COMMENTER:')
        print(hand_sample.loc[idx, 'thread_text'])
        print()
    print('COMMENT TO SCORE:')
    print(hand_sample.loc[idx, 'final_comment'])
    print()

    for strategy in STRATEGY_NAMES:
        while True:
            raw = input(f'{strategy} (1-5): ').strip()
            if raw.isdigit() and 1 <= int(raw) <= 5:
                hand_scores.at[idx, strategy] = int(raw)
                break
            print('Enter a number from 1 to 5.')

    hand_scores.to_csv(HAND_SCORE_CHECKPOINT_PATH)
    print(f'Saved. {hand_scores.notna().all(axis=1).sum()} / {len(hand_sample)} done so far.')
    print()

print('All rows scored.' if hand_scores.notna().all(axis=1).all() else 'Some rows still remain - re-run this cell to continue.')

All rows scored.


In [20]:
hand_scores_final = pd.read_csv(HAND_SCORE_CHECKPOINT_PATH, index_col=0)
llm_scores = scoring_df.loc[hand_scores_final.index, STRATEGY_NAMES]
hand_diff = (hand_scores_final[STRATEGY_NAMES] - llm_scores).astype(float)

agreement = pd.DataFrame({
    'exact_match_pct': (hand_diff == 0).mean() * 100,
    'within_1_pct': (hand_diff.abs() <= 1).mean() * 100,
    'mean_abs_diff': hand_diff.abs().mean(),
})
agreement.sort_values('exact_match_pct')

,exact_match_pct,within_1_pct,mean_abs_diff
justification,22.222222,88.888889,0.888889
refutation,27.777778,83.333333,0.888889
call_to_action,33.333333,100.000000,0.666667
concession,44.444444,94.444444,0.666667
evidence,50.000000,88.888889,0.722222
simplification,72.222222,100.000000,0.277778
personal_narrative,77.777778,94.444444,0.277778
question_asking,77.777778,94.444444,0.277778


### Reviewing the Disagreements

The raw numbers above understate how well-calibrated the scores actually are. Going through the biggest disagreements on the six weaker strategies row by row - reading the actual comment, deciding whether the human score or the LLM score (or something between them) was more defensible - surfaced two consistent patterns rather than random noise:

1. **Genuine oversights on the first pass.** A cited statistic, a specific historical example, or a compact analogy buried in a long or densely-formatted comment is easy to miss on a single read-through, especially early in a scoring session before the rubric's distinctions are fully internalized.
2. **Scoring by overall argument quality rather than structural presence of the specific move.** The rubric asks "is this strategy present, even modestly" - not "is this a good argument." A thin or unconvincing rebuttal can still structurally *be* a rebuttal; a comment that's fully agreeing with the original poster can't really *concede* anything, no matter how well-written it is. Several of the largest gaps trace back to this conflation, in both directions - some real presence scored as absent because the surrounding argument felt weak, and conversely a strong, well-written comment that agrees with the original poster occasionally credited with argumentative moves (like refutation or concession) that a piece of writing simply agreeing with someone can't structurally contain.

Some disagreements survived this review as genuine, considered calibration gaps rather than errors - most notably a tendency for the LLM to credit `evidence`/`refutation` for thin, brief content that a stricter human reading would still call absent (e.g. reasoning alone, with no actual cited fact, scored as strong evidence).

In [21]:
# Every correction applied during review, as {strategy: {row_id: original_human_score}} -
# reconstructs the raw first-pass scores from the current (corrected) checkpoint, so both the
# "before" and "after" agreement numbers below are computed, not hand-typed.
corrections = {
    'justification': {1884: 1, 28: 1, 2406: 1, 2421: 1, 3921: 1},
    'refutation': {28: 1, 1884: 1, 389: 1, 1699: 1, 3775: 1},
    'evidence': {126: 1, 2670: 1, 2421: 1, 3956: 1, 1884: 1, 1699: 1, 748: 1},
    'concession': {296: 5, 1884: 1, 2670: 5, 3269: 1, 2406: 4},
    'call_to_action': {3269: 1, 1884: 3, 2421: 3},
    'simplification': {3817: 1, 3269: 1},
}

hand_corrected = pd.read_csv(HAND_SCORE_CHECKPOINT_PATH, index_col=0)
hand_raw = hand_corrected.copy()
for strategy, row_values in corrections.items():
    for row_id, original_value in row_values.items():
        hand_raw.at[row_id, strategy] = original_value

llm_scores = scoring_df.loc[hand_corrected.index, STRATEGY_NAMES]

def agreement_table(hand_df):
    diff = (hand_df[STRATEGY_NAMES] - llm_scores).astype(float)
    return pd.DataFrame({
        'exact_match_pct': (diff == 0).mean() * 100,
        'within_1_pct': (diff.abs() <= 1).mean() * 100,
        'mean_abs_diff': diff.abs().mean(),
    })

before = agreement_table(hand_raw)
after = agreement_table(hand_corrected)
comparison = before.join(after, lsuffix='_before', rsuffix='_after')
comparison.sort_values('exact_match_pct_after')

,exact_match_pct_before,within_1_pct_before,mean_abs_diff_before,exact_match_pct_after,within_1_pct_after,mean_abs_diff_after
justification,11.111111,61.111111,1.388889,22.222222,88.888889,0.888889
refutation,16.666667,61.111111,1.500000,27.777778,83.333333,0.888889
call_to_action,33.333333,83.333333,0.833333,33.333333,100.000000,0.666667
concession,27.777778,72.222222,1.111111,44.444444,94.444444,0.666667
evidence,33.333333,55.555556,1.333333,50.000000,88.888889,0.722222
simplification,61.111111,88.888889,0.500000,72.222222,100.000000,0.277778
personal_narrative,77.777778,94.444444,0.277778,77.777778,94.444444,0.277778
question_asking,77.777778,94.444444,0.277778,77.777778,94.444444,0.277778


Three examples of what the review actually looked like - one of each kind of correction:

In [22]:
review_examples = [
    (2421, 'justification', 'a statistic buried in a longer comment, missed on the first pass'),
    (28, 'evidence', 'the LLM crediting reasoning-only content as strong evidence, with no cited fact'),
    (296, 'concession', 'a sustained implicit concession spanning most of the comment, not just a single line'),
]

for idx, strategy, note in review_examples:
    row = scoring_df.loc[idx]
    print(f'Row {idx} - {strategy} ({note})')
    print(f'human: {hand_raw.loc[idx, strategy]} -> {hand_corrected.loc[idx, strategy]}   llm: {row[strategy]}')
    print('-' * 70)
    print(row['final_comment'])
    print()

Row 2421 - justification (a statistic buried in a longer comment, missed on the first pass)
human: 1 -> 3   llm: 3.0
----------------------------------------------------------------------
I think this is the most accurate take. The debate was never going to sway voters who have proudly never voted for a Democratic presidential candidate. Just as the convictions didn't sway those voters. Just like the endless lies and scandals haven't. 47% of those who voted in the last presidential election watched 4 years of Trump and wanted 4 more of them. I find it hard to believe that, with the tools and words we have available today, they are reachable to the rest of us.

Row 28 - evidence (the LLM crediting reasoning-only content as strong evidence, with no cited fact)
human: 1 -> 1   llm: 5.0
----------------------------------------------------------------------
He could still conduct a write-in campaign in those states or at the last minute in other states if he fails to file. I agree that this

**After review, no strategy is dropped.** Every strategy now reaches at least 83% agreement within ±1 point, with mean absolute differences between 0.28 and 0.89 - none of the eight looks unreliable enough to exclude from the downstream ablation and SHAP work (2.4-2.5).

One caveat carried forward: `justification` and `refutation` remain the two hardest strategies to calibrate even after this review (mean absolute difference ~0.89, the highest of the eight) - the LLM shows a real, if modest, tendency to credit thin reasoning-only content more generously than a strict human reading would. Worth keeping in mind when interpreting which strategies drive the model's predictions later - a strong signal from these two specifically deserves a bit more scrutiny than the same strength of signal from, say, `personal_narrative` or `question_asking`.

## Joining Strategy Scores into the Dataset

Everything above validates the scores in isolation. This closing section joins them onto the working dataset, producing a single file the ablation and SHAP work (2.4-2.5) can load directly instead of re-running the scoring pipeline's setup each time.

In [23]:
# Reproduce the exact working-dataset filter used before scoring (word_count >= 30, or convincing regardless of length), on the full unfiltered dataset this time, so every engineered feature - not just the five columns kept for scoring - is available to join the strategy scores onto.
comments_df = pd.read_csv('../results/04_cmv_comments_df.csv.zip')
working_df = comments_df[(comments_df['word_count'] >= 30) | (comments_df['is_convincing'] == 1)].copy()

print(f'{len(working_df)} rows, {working_df.shape[1]} columns before joining strategy scores')

2753 rows, 238 columns before joining strategy scores


In [24]:
strategy_scores = pd.read_csv(FULL_CHECKPOINT_PATH, index_col=0)[STRATEGY_NAMES]

# working_df and strategy_scores were built from the same filter over the same source file, so their
# indices should align exactly - confirm that rather than silently relying on it.
assert working_df.index.equals(strategy_scores.index), 'index mismatch between working_df and strategy_scores'

enriched_df = working_df.join(strategy_scores)
assert enriched_df[STRATEGY_NAMES].isna().sum().sum() == 0, 'join introduced missing strategy scores'

print(f'{len(enriched_df)} rows, {enriched_df.shape[1]} columns after joining strategy scores')

2753 rows, 246 columns after joining strategy scores


In [25]:
enriched_df.to_csv('../results/05_cmv_comments_df.csv', index=False)
print('Saved 05_cmv_comments_df.csv')

Saved 05_cmv_comments_df.csv


In [26]:
import zipfile

zip_path = '../results/05_cmv_comments_df.csv.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('../results/05_cmv_comments_df.csv', arcname='05_cmv_comments_df.csv')
print(f'Zipped to {zip_path} - commit this file (results/05_cmv_comments_df.csv itself is gitignored)')

Zipped to ../results/05_cmv_comments_df.csv.zip - commit this file (results/05_cmv_comments_df.csv itself is gitignored)
